In [ ]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

from tqdm import tqdm 

from loka.data import load_breakthrough_states, save_breakthrough_states, load_forgetting_states, save_forgetting_states
from loka.models import BreakthroughCNN
from loka.training_methods import ForgettingScores

import os
import pandas as pd
from dotenv import load_dotenv

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

load_dotenv()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

BASE = os.getenv("BASE_DIR")

X,Y = load_breakthrough_states(f"{BASE}/data/game_states/e.txt")
print(X.shape)

In [ ]:
epochs = 200

pbar = tqdm(
    total=epochs,
    desc="training",
    bar_format="{l_bar}{bar} {percentage:3.0f}% [{elapsed}<{remaining}]",
)

X_t, Y_t = X[:100], Y[:100]

torch.manual_seed(SEED)
model = BreakthroughCNN()
forgetter = ForgettingScores(model, seed=SEED)

forgetter.train(
    X = X_t,
    Y = Y_t,
    epochs = epochs,
    batch_size= 512,
    random_permuations=True,
    device=device,
    pbar=pbar,
    pbar_increment=1
)


In [ ]:
C= forgetter.getForgettingCounts().cpu()
C.shape

In [ ]:
dirname = f"{BASE}/data/forgetting_counts"
os.makedirs(dirname, exist_ok=True)

fname = f"{dirname}/forgetting_e.txt"
save_forgetting_states(fname, X[:100],Y[:100],C)